# Turbulence Metric Surrogate

## What you will learn
How optimizing a proxy can select a design that fails nonlinear validation.

## Codes used
Pure Python cached design table; optional SPECTRAX-GK research path.

## Run mode
This notebook uses RUN_MODE = "cached" by default. Allowed values are "tiny", "cached", and "research".

## Expected outputs
`09_proxy_vs_nonlinear.png` and `09_w7x_clamping_cartoon.png`.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "src" / "sos2026").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    print("Colab detected. Keep RUN_MODE='cached' first; install requirements-colab.txt from the cloned repo if needed.")
else:
    print("Local runtime detected.")

In [ ]:
RUN_MODE = "cached"  # allowed: "tiny", "cached", "research"
print(f"RUN_MODE = {RUN_MODE}")

In [ ]:
import importlib
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sos2026.paths import PROJECT_ROOT, DATA_DIR, CACHE_DIR, FIGURE_DIR, MOVIE_DIR, ensure_directories
ensure_directories()
print("Figures:", FIGURE_DIR)
print("Cached data:", CACHE_DIR)

## 1. Learning frame

This notebook is a deliberately small project: define one metric, produce one plot, expose one failure mode, and identify where a real code would enter.

In [ ]:
from sos2026.turbulence_helpers import proxy_validation_table, clamping_cartoon
from sos2026.plotting import savefig, caption

## 2. Load or generate the teaching data

Cached mode uses small arrays so the conceptual workflow is always available.

In [ ]:
table = proxy_validation_table()
table

## 3. Make the primary plot

Every plot has a one-sentence caption because students should know how to read it without guessing.

In [ ]:
fig, ax = plt.subplots(figsize=(6.8, 3.9))
x = np.arange(len(table))
for col in ["linear_proxy", "quasilinear_proxy", "nonlinear_validation"]:
    ax.plot(x, table[col], marker="o", lw=2, label=col)
ax.set_xticks(x, table["design"], rotation=25, ha="right")
ax.set_ylabel("normalized heat-flux-like metric")
ax.set_title("Proxy vs nonlinear validation")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
caption(ax, "Different metrics rank the same designs differently; validation gates matter.")
savefig(fig, "09_proxy_vs_nonlinear.png")
plt.show()

## 4. Probe the metric

A metric becomes useful for optimization only when we understand how it changes across design choices.

In [ ]:
power, neo, turb = clamping_cartoon()
fig, ax = plt.subplots(figsize=(6.0, 3.7))
ax.plot(power, neo, lw=2, label="neoclassical-limited expectation")
ax.plot(power, turb, lw=2, label="turbulence-clamped cartoon")
ax.set_xlabel("heating power proxy")
ax.set_ylabel("ion-temperature proxy")
ax.set_title("W7-X ion-temperature clamping cartoon")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
caption(ax, "A design can pass neoclassical gates and still need turbulence-aware validation.")
savefig(fig, "09_w7x_clamping_cartoon.png")
plt.show()

## 5. Interpret the design consequence

The table below translates the plot into an optimization decision.

In [ ]:
print("Best by linear proxy:", table.loc[table.linear_proxy.idxmin(), "design"])
print("Best by nonlinear validation:", table.loc[table.nonlinear_validation.idxmin(), "design"])
table.assign(linear_rank=table.linear_proxy.rank(), nonlinear_rank=table.nonlinear_validation.rank())

## 6. Failure mode

The cached plot is useful only if we say what it does not prove.

In [ ]:
failure_mode = pd.DataFrame({
    "cached_mode_proves": ["workflow shape", "plot grammar", "where the metric enters"],
    "cached_mode_does_not_prove": ["validated physics", "final design ranking", "runtime scalability"],
})
failure_mode

## 7. Research-mode hook

Run this cell only after timing the package on the lecture machine.

In [ ]:
if RUN_MODE == "research":
    print("Research path: replace nonlinear_validation with a validated nonlinear output column.")
else:
    print("Cached mode: research package path skipped intentionally.")

## 8. Mini project handoff

Use this notebook during the lecture as the computational project slide points to: change one parameter, regenerate one plot, and explain one design tradeoff.

In [ ]:
project_steps = pd.DataFrame({
    "step": [1, 2, 3, 4],
    "action": ["identify metric", "change one input", "regenerate plot", "state failure mode"],
})
project_steps

## Try this
Change one scalar or one row in the cached data and regenerate the primary plot.

## Expected qualitative answer
The plot should move in a physically interpretable direction, but the cached result remains an educational proxy.

## Research extension
Replace the cached data source with the corresponding real package output after timing and API verification.